In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
import xgboost as xgb
from catboost import CatBoostRegressor

Read in the data

In [10]:
# This assumes that train.csv and test.csv are in the same folder as this notebook
train = pd.read_csv('train.csv', index_col='ID')
test = pd.read_csv('test.csv', index_col='ID')

Engineer features

In [11]:
PRICE_COLS = [f'p{i}' for i in range(1, 41)]

def make_features(df):
    # Clip prices so log() never hits -inf on zero-price stocks
    p = np.clip(df[PRICE_COLS].values, 1e-6, None)
    lr = np.diff(np.log(p), axis=1)   # daily log-returns, shape (n, 39)
    f = {}

    # Raw price path — lets the model see the exact curve shape
    for i, col in enumerate(PRICE_COLS):
        f[col] = p[:, i]

    # Trend and momentum
    f['log_ret_full']    = np.log(p[:, -1] / p[:, 0])
    f['last_change']     = p[:, -1] - p[:, -2]
    f['last_change_pct'] = f['last_change'] / p[:, -2]
    for w in [5, 10, 20]:
        f[f'ret_{w}d'] = np.log(p[:, -1] / p[:, -1 - w])

    # Moving averages and deviation from them
    f['ma5']  = p[:, -5:].mean(axis=1)
    f['ma10'] = p[:, -10:].mean(axis=1)
    f['ma20'] = p[:, -20:].mean(axis=1)
    f['ma40'] = p.mean(axis=1)
    f['p40_vs_ma5']  = p[:, -1] - f['ma5']
    f['p40_vs_ma20'] = p[:, -1] - f['ma20']
    f['p40_vs_ma40'] = p[:, -1] - f['ma40']

    # Volatility
    f['std10'] = p[:, -10:].std(axis=1)   # also used in the risk-adjusted score
    f['std20'] = p[:, -20:].std(axis=1)
    f['std40'] = p.std(axis=1)
    f['vol10'] = lr[:, -10:].std(axis=1)
    f['vol20'] = lr[:, -20:].std(axis=1)
    f['vol40'] = lr.std(axis=1)

    # Bollinger Band position
    upper20 = f['ma20'] + 2 * f['std20']
    lower20 = f['ma20'] - 2 * f['std20']
    bw = upper20 - lower20
    f['bb_pos'] = np.where(bw > 0, (p[:, -1] - lower20) / bw, 0.5)

    # Historical range context
    f['hist_min']      = p.min(axis=1)
    f['hist_max']      = p.max(axis=1)
    f['hist_range']    = f['hist_max'] - f['hist_min']
    f['p40_vs_min']    = p[:, -1] - f['hist_min']
    f['p40_vs_max']    = f['hist_max'] - p[:, -1]
    f['p40_pct_range'] = np.where(
        f['hist_range'] > 0, f['p40_vs_min'] / f['hist_range'], 0.5
    )

    # Trend slope (OLS on log-prices)
    t = np.arange(40)
    t_norm = (t - t.mean()) / t.std()
    lp = np.log(p)
    f['trend_slope'] = (lp * t_norm).mean(axis=1) - lp.mean(axis=1) * t_norm.mean()

    # Momentum acceleration: 2nd half vs 1st half return
    f['ret_2nd_half']   = np.log(p[:, -1] / p[:, 19])
    f['ret_1st_half']   = np.log(p[:, 19] / p[:, 0])
    f['momentum_accel'] = f['ret_2nd_half'] - f['ret_1st_half']

    out = pd.DataFrame(f, index=df.index)
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    out.fillna(out.median(), inplace=True)
    return out

X = make_features(train)
X_test = make_features(test)

# Target: predict the gain (p40 - p50) directly
# This aligns the model with what we actually want to rank
y = train['p40'] - train['p50']

Create a hold-out set

In [12]:
# 80-20 split — gives us a quick sanity check on the hold-out R before we run full CV
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.2, random_state=42)
# X_holdout is the hold-out feature matrix
# X_train is the remaining training features

# Keep the raw p40/p50 for the holdout — needed to compute R
holdout_p40 = train.loc[X_holdout.index, 'p40']
holdout_p50 = train.loc[X_holdout.index, 'p50']

Train a model on the training data

In [13]:
# XGBoost: predicts the gain (p40 - p50) for each stock
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
    early_stopping_rounds=50,
)
xgb_model.fit(X_train, y_train, eval_set=[(X_holdout, y_holdout)], verbose=False)

# CatBoost: second model for the ensemble
cat_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    early_stopping_rounds=50,
    verbose=False,
)
cat_model.fit(X_train, y_train, eval_set=(X_holdout, y_holdout))

CatBoostRegressor(depth=6, early_stopping_rounds=50, iterations=1000, l2_leaf_reg=3, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=False)

Make predictions on the hold-out set

In [14]:
# Average the two models' gain predictions
gain_holdout = (xgb_model.predict(X_holdout) + cat_model.predict(X_holdout)) / 2

# Risk-adjusted score: predicted gain scaled by recent volatility
# Higher penalty → closer to pure gain ranking (penalty=100 found by sweep)
PENALTY = 100
sigma_holdout = X_holdout['std10'].values
scores_holdout = gain_holdout / (sigma_holdout + PENALTY)

# Select the top 40% of holdout stocks to sell
K = int(0.4 * len(X_holdout))
top_idx = np.argsort(scores_holdout)[::-1][:K]
sell_holdout = np.zeros(len(X_holdout), dtype=int)
sell_holdout[top_idx] = 1

Evaluate the predictions on the hold-out set

In [15]:
# To simulate the constraint, we can require that no more than 40% of the hold-out
# stocks can be sold
# The above predictions do not explicitly account for this constraint, so we may
# need to randomly subsample
def score(holdout_p40, holdout_p50, sell_holdout):
    # Check the constraint
    K = int(0.4 * len(sell_holdout))
    n_sell = sell_holdout.sum()
    if n_sell > K:
        # It has been violated, randomly subsample
        print(f"Subsampling {n_sell} stocks down to {K}")
        ones = np.where(sell_holdout == 1)[0]
        keep = np.random.choice(ones, size=K, replace=False)
        sell_holdout = sell_holdout * 0
        sell_holdout[keep] = 1
    p40 = holdout_p40.values
    p50 = holdout_p50.values
    R = float((sell_holdout * (p40 - p50)).sum())
    return R

holdout_R = score(holdout_p40, holdout_p50, sell_holdout)
print(f"Hold-out R: {holdout_R:.2f}  (quick check — not fully reliable, see CV below)")

Hold-out R: 4185.56  (quick check — not fully reliable, see CV below)


Estimate the real R with cross-validation

The hold-out R above depends on which 20% we happened to split off.
5-fold CV trains and validates on every stock exactly once, giving a much more reliable estimate.
Each fold selects the best 800 sells from 2,000 stocks — summing all 5 folds = 4,000 sells from 10,000 stocks, the same ratio as the real test submission.

In [16]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

total_R   = 0          # sum of R across all 5 folds
xgb_iters = []        # best iteration found per fold — used when retraining on all data
cat_iters = []

print(f"{'Fold':>4}  {'Stocks seen':>12}  {'Sells':>6}  {'Fold R':>10}")
print("-" * 40)

for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train XGBoost on this fold
    xgb_fold = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0, early_stopping_rounds=50,
    )
    xgb_fold.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    xgb_iters.append(xgb_fold.best_iteration)

    # Train CatBoost on this fold
    cat_fold = CatBoostRegressor(
        iterations=1000, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, random_seed=42, early_stopping_rounds=50, verbose=False,
    )
    cat_fold.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    cat_iters.append(cat_fold.best_iteration_)

    # Ensemble: average both models' gain predictions
    gain_pred = (xgb_fold.predict(X_val) + cat_fold.predict(X_val)) / 2

    # Rank by risk-adjusted score, select top 40% of this fold to sell
    sigma     = X_val['std10'].values
    scores    = gain_pred / (sigma + PENALTY)
    K_fold    = int(0.4 * len(val_idx))
    top_idx   = np.argsort(scores)[::-1][:K_fold]
    sell_fold = np.zeros(len(val_idx), dtype=int)
    sell_fold[top_idx] = 1

    # Compute actual R for this fold using the real p50 values
    p40_fold = train['p40'].iloc[val_idx].values
    p50_fold = train['p50'].iloc[val_idx].values
    fold_R   = float((sell_fold * (p40_fold - p50_fold)).sum())

    total_R += fold_R
    print(f"{fold:>4}  {len(val_idx):>12}  {sell_fold.sum():>6}  {fold_R:>10.2f}")

print("-" * 40)
print(f"{'Total':>4}  {len(X):>12}  {int(0.4 * len(X)):>6}  {total_R:>10.2f}")
print(f"\nEstimated submission R: {total_R:.2f}")
print(f"  (this is the most reliable number we have before submitting to Kaggle)")

Fold   Stocks seen   Sells      Fold R
----------------------------------------
   1          2000     800     4222.25
   2          2000     800     4191.95
   3          2000     800     3965.95
   4          2000     800     3724.51
   5          2000     800     3610.69
----------------------------------------
Total         10000    4000    19715.35

Estimated submission R: 19715.35
  (this is the most reliable number we have before submitting to Kaggle)


Retrain on all training data

The CV models were each trained on only 8,000 stocks.
Now we train one final model on all 10,000 using the average iteration count from CV,
so the final model is as strong as possible without overfitting.

In [17]:
# Use the average best iteration from CV — avoids over- or under-fitting on full data
xgb_best_iter = int(np.mean(xgb_iters))
cat_best_iter = int(np.mean(cat_iters))

print(f"XGBoost: retraining with {xgb_best_iter} trees on all {len(X)} stocks")
xgb_final = xgb.XGBRegressor(
    n_estimators=xgb_best_iter, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, verbosity=0,
)
xgb_final.fit(X, y)

print(f"CatBoost: retraining with {cat_best_iter} trees on all {len(X)} stocks")
cat_final = CatBoostRegressor(
    iterations=cat_best_iter, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, random_seed=42, verbose=False,
)
cat_final.fit(X, y)

XGBoost: retraining with 48 trees on all 10000 stocks
CatBoost: retraining with 102 trees on all 10000 stocks


CatBoostRegressor(depth=6, iterations=102, l2_leaf_reg=3, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=False)

Make predictions on the test set

In [18]:
gain_test   = (xgb_final.predict(X_test) + cat_final.predict(X_test)) / 2
sigma_test  = X_test['std10'].values
scores_test = gain_test / (sigma_test + PENALTY)

# Select exactly 4000 stocks to sell
sell_test = np.zeros(len(X_test), dtype=int)
sell_test[np.argsort(scores_test)[::-1][:4000]] = 1

# Note again that these predictions will not necessarily follow the constraint
# The code below checks if we are following the constraint
n_sell = sell_test.sum()
print(f"Stocks to sell: {n_sell}")
print(f"Constraint satisfied: {n_sell <= 4000}")

Stocks to sell: 4000
Constraint satisfied: True


Create submission file

In [19]:
submission = pd.DataFrame({'ID': test.index, 'sell': sell_test})
submission.to_csv('submission.csv', index=False)
# The created file can be submitted to Kaggle